# 02_extract_with_deepseek_final_json_fallback20

Второй этап пайплайна.

Этот ноутбук читает `chunks.json`, отправляет чанки в DeepSeek API и сохраняет итоговый JSON.

Основная стратегия:

```text
один лист Excel = один чанк
```

Резервная стратегия:

```text
если лист целиком не обработался,
то проблемный лист автоматически делится на подчанки по 20 строк
```

Итог:

```text
chunks.json → DeepSeek API → requirements_output_deepseek.json
```

## 1. Установка библиотек

In [1]:
%pip install requests python-dotenv

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\Artem\AppData\Local\Programs\Python\Python39\python.exe -m pip install --upgrade pip' command.


## 2. Импорты и настройки

In [2]:
from pathlib import Path
import json
import os
import time
import requests
from dotenv import load_dotenv


load_dotenv()

CHUNKS_INPUT = Path("chunks.json")

JSON_OUTPUT = Path("requirements_output_deepseek.json")
JSON_CHECKPOINT = Path("requirements_output_deepseek_checkpoint.json")

DEEPSEEK_API_URL = "https://api.deepseek.com/chat/completions"
MODEL_NAME = "deepseek-v4-flash"

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")

# Основная стратегия: один лист = один чанк.
PRIMARY_CHUNKING_STRATEGY = "one_sheet_one_chunk"

# Резервная стратегия: если лист не обработался, делим его на подчанки.
FALLBACK_ROWS_PER_CHUNK = 5

# Хардкод по требованиям проекта.
DOCUMENT_CATEGORY_ID = 6
SYSTEM_PROMPT_ID = 1
VERSION = 1
STATUS = "published"

print("CHUNKS_INPUT:", CHUNKS_INPUT)
print("JSON_OUTPUT:", JSON_OUTPUT)
print("MODEL_NAME:", MODEL_NAME)
print("PRIMARY_CHUNKING_STRATEGY:", PRIMARY_CHUNKING_STRATEGY)
print("FALLBACK_ROWS_PER_CHUNK:", FALLBACK_ROWS_PER_CHUNK)
print("DOCUMENT_CATEGORY_ID:", DOCUMENT_CATEGORY_ID)
print("SYSTEM_PROMPT_ID:", SYSTEM_PROMPT_ID)
print("VERSION:", VERSION)
print("STATUS:", STATUS)

CHUNKS_INPUT: chunks.json
JSON_OUTPUT: requirements_output_deepseek.json
MODEL_NAME: deepseek-v4-flash
PRIMARY_CHUNKING_STRATEGY: one_sheet_one_chunk
FALLBACK_ROWS_PER_CHUNK: 5
DOCUMENT_CATEGORY_ID: 6
SYSTEM_PROMPT_ID: 1
VERSION: 1
STATUS: published


## 3. Проверка DeepSeek API key из `.env`

In [3]:
if not DEEPSEEK_API_KEY:
    raise RuntimeError(
        "Не найден DEEPSEEK_API_KEY. "
        "Создай файл .env рядом с ноутбуком и добавь строку: "
        "DEEPSEEK_API_KEY=твой_ключ"
    )

print("DeepSeek API key найден.")

DeepSeek API key найден.


## 4. Проверка файла `chunks.json`

In [4]:
if not CHUNKS_INPUT.exists():
    raise FileNotFoundError(
        f"Файл не найден: {CHUNKS_INPUT}. "
        "Сначала запусти первый ноутбук, который создаёт chunks.json."
    )

chunks = json.loads(CHUNKS_INPUT.read_text(encoding="utf-8"))

print(f"Загружено чанков: {len(chunks)}")

for chunk in chunks[:12]:
    print(
        f"chunk={chunk.get('chunk_id')} | "
        f"section={chunk.get('source_section')} | "
        f"sheet={chunk.get('sheet_name')} | "
        f"rows_count={chunk.get('rows_count')} | "
        f"strategy={chunk.get('chunking_strategy')} | "
        f"chars={len(chunk.get('text', ''))}"
    )

Загружено чанков: 12
chunk=1 | section=поставка, купля-прод. | sheet=поставка, купля-прод. | rows_count=29 | strategy=one_sheet_one_chunk | chars=22161
chunk=2 | section=подряд | sheet=подряд | rows_count=20 | strategy=one_sheet_one_chunk | chars=26512
chunk=3 | section=ГПД с ФЛ | sheet=ГПД с ФЛ | rows_count=18 | strategy=one_sheet_one_chunk | chars=15251
chunk=4 | section=услуги | sheet=услуги | rows_count=18 | strategy=one_sheet_one_chunk | chars=21819
chunk=5 | section=аренда | sheet=аренда | rows_count=18 | strategy=one_sheet_one_chunk | chars=22560
chunk=6 | section=хранение | sheet=хранение | rows_count=16 | strategy=one_sheet_one_chunk | chars=23887
chunk=7 | section=перевозка, экспедиция | sheet=перевозка, экспедиция | rows_count=20 | strategy=one_sheet_one_chunk | chars=25221
chunk=8 | section=лицензионный | sheet=лицензионный | rows_count=18 | strategy=one_sheet_one_chunk | chars=20189
chunk=9 | section=страхование | sheet=страхование | rows_count=12 | strategy=one_sheet_one_

## 5. Функция запроса к DeepSeek

In [5]:
def ask_deepseek(prompt: str, model: str = MODEL_NAME, timeout: int = 600) -> str:
    """
    Отправляет prompt в DeepSeek API.

    Используется OpenAI-compatible endpoint:
    POST https://api.deepseek.com/chat/completions
    """

    headers = {
        "Authorization": f"Bearer {DEEPSEEK_API_KEY}",
        "Content-Type": "application/json",
    }

    payload = {
        "model": model,
        "messages": [
            {
                "role": "system",
                "content": (
                    "Ты система извлечения требований из табличных документов. "
                    "Отвечай только валидным JSON без Markdown и без пояснений."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        "temperature": 0,
        "stream": False,
        "max_tokens": 20000,
        "thinking": {"type": "enabled"},
        "reasoning_effort": "high",
        "response_format": {
            "type": "json_object"
        }
    }

    response = requests.post(
        DEEPSEEK_API_URL,
        headers=headers,
        json=payload,
        timeout=timeout
    )

    response.raise_for_status()
    data = response.json()

    return data["choices"][0]["message"]["content"]


test_answer = ask_deepseek(
    'Верни JSON: {"status": "ok"}',
    timeout=120
)

print("Ответ DeepSeek:")
print(test_answer)

Ответ DeepSeek:
{"status": "ok"}


## 6. Парсинг JSON и нормализация

In [6]:
def extract_json_object_from_text(text: str) -> dict:
    """Пытается достать JSON-объект из ответа модели."""

    original_text = text
    text = text.strip()

    if text.startswith("```json"):
        text = text.replace("```json", "", 1).strip()

    if text.startswith("```"):
        text = text.replace("```", "", 1).strip()

    if text.endswith("```"):
        text = text[:-3].strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    start = text.find("{")
    end = text.rfind("}")

    if start == -1 or end == -1 or end <= start:
        raise ValueError("JSON-объект не найден в ответе модели")

    json_text = text[start:end + 1]

    try:
        return json.loads(json_text)
    except json.JSONDecodeError as exc:
        debug_path = Path("debug_bad_deepseek_response.json.txt")
        debug_path.write_text(original_text, encoding="utf-8")

        raise ValueError(
            f"DeepSeek вернул невалидный JSON. "
            f"Сырой ответ сохранён в {debug_path}. "
            f"Ошибка: {exc}"
        )


def clean_text(value) -> str:
    """Безопасно превращает значение в строку."""

    if value is None:
        return ""

    text = str(value).strip()

    # Убираем лишние переносы, чтобы итоговый JSON был компактнее.
    text = text.replace("\r\n", " ").replace("\n", " ").replace("\r", " ")
    text = " ".join(text.split())

    return text

def normalize_requirement(item: dict, source_chunk: dict) -> dict:
    """
    Приводит одно требование к финальной схеме.

    Технические поля заполняются жёстко:
    - document_category_id = 6
    - system_prompt_id = 1
    - version = 1
    - status = published
    """

    return {
        "name": clean_text(item.get("name")),
        "requirements_brief": clean_text(item.get("requirements_brief")),
        "check_prompt": clean_text(item.get("check_prompt")),
        "document_category_id": DOCUMENT_CATEGORY_ID,
        "system_prompt_id": SYSTEM_PROMPT_ID,
        "version": VERSION,
        "status": STATUS,
        "risks": clean_text(item.get("risks")),
        "severity": clean_text(item.get("severity")),
        "check_result_example": clean_text(item.get("check_result_example")),
        "suggested_solutions": clean_text(item.get("suggested_solutions")),
        "source": source_chunk,
    }


def validate_requirements(data: dict, source_chunk: dict) -> list[dict]:
    """Проверяет и нормализует список требований."""

    if not isinstance(data, dict):
        raise TypeError("Ответ модели должен быть JSON-объектом")

    items = data.get("requirements", [])

    if not isinstance(items, list):
        raise TypeError("Поле requirements должно быть списком")

    result = []

    for item in items:
        if not isinstance(item, dict):
            continue

        req = normalize_requirement(item, source_chunk)

        if req["name"] or req["requirements_brief"] or req["check_prompt"]:
            result.append(req)

    return result

## 7. Деление проблемного листа на fallback-подчанки

In [7]:
def split_markdown_table_chunk_by_rows(chunk: dict, rows_per_chunk: int = FALLBACK_ROWS_PER_CHUNK) -> list[dict]:
    """
    Делит Markdown-таблицу внутри одного листа на подчанки по N строк.

    Вход:
    {
      "chunk_id": 7,
      "source_section": "...",
      "sheet_name": "...",
      "rows_count": 20,
      "text": "# Лист: ...\\n| header | ..."
    }

    Выход:
    [
      {
        "chunk_id": "7.1",
        "parent_chunk_id": 7,
        "source_section": "...",
        "sheet_name": "...",
        "row_start": 1,
        "row_end": 20,
        "rows_count": 20,
        "chunking_strategy": "fallback_rows_20",
        "text": "..."
      }
    ]
    """

    text = chunk.get("text", "")
    lines = text.splitlines()

    table_lines = [line for line in lines if line.strip().startswith("|")]
    intro_lines = [line for line in lines if not line.strip().startswith("|")]

    if len(table_lines) < 3:
        # Если таблица не распозналась, возвращаем исходный чанк как один fallback.
        fallback_chunk = dict(chunk)
        fallback_chunk["parent_chunk_id"] = chunk.get("chunk_id")
        fallback_chunk["chunk_id"] = f"{chunk.get('chunk_id')}.1"
        fallback_chunk["chunking_strategy"] = f"fallback_rows_{rows_per_chunk}"
        return [fallback_chunk]

    header_line = table_lines[0]
    separator_line = table_lines[1]
    row_lines = table_lines[2:]

    fallback_chunks = []

    for start in range(0, len(row_lines), rows_per_chunk):
        end = min(start + rows_per_chunk, len(row_lines))
        part_rows = row_lines[start:end]

        part_text_lines = []
        part_text_lines.extend(intro_lines)
        part_text_lines.append("")
        part_text_lines.append(f"Fallback rows: {start + 1}-{end}")
        part_text_lines.append("")
        part_text_lines.append(header_line)
        part_text_lines.append(separator_line)
        part_text_lines.extend(part_rows)

        fallback_chunks.append(
            {
                "chunk_id": f"{chunk.get('chunk_id')}.{len(fallback_chunks) + 1}",
                "parent_chunk_id": chunk.get("chunk_id"),
                "source_section": chunk.get("source_section"),
                "sheet_name": chunk.get("sheet_name"),
                "row_start": start + 1,
                "row_end": end,
                "rows_count": end - start,
                "chunking_strategy": f"fallback_rows_{rows_per_chunk}",
                "text": "\n".join(part_text_lines),
            }
        )

    return fallback_chunks


# Быстрая проверка fallback на первом чанке
example_fallback = split_markdown_table_chunk_by_rows(chunks[0], rows_per_chunk=FALLBACK_ROWS_PER_CHUNK)
print("Fallback-подчанков для первого чанка:", len(example_fallback))
for sub in example_fallback[:3]:
    print(sub["chunk_id"], sub["row_start"], sub["row_end"], sub["rows_count"], len(sub["text"]))

Fallback-подчанков для первого чанка: 6
1.1 1 5 5 3310
1.2 6 10 5 2933
1.3 11 15 5 3153


## 8. Prompt для извлечения требований

In [8]:
def build_prompt_for_chunk(chunk: dict) -> str:
    """Создаёт prompt для одного чанка."""

    return f"""
Ты извлекаешь требования из Excel-таблицы.

Источник:
- chunk_id: {chunk.get("chunk_id")}
- parent_chunk_id: {chunk.get("parent_chunk_id", "")}
- source_section: {chunk.get("source_section")}
- sheet_name: {chunk.get("sheet_name")}
- rows_count: {chunk.get("rows_count")}
- row_start: {chunk.get("row_start", "")}
- row_end: {chunk.get("row_end", "")}
- chunking_strategy: {chunk.get("chunking_strategy")}

Нужно извлечь требования из строк таблицы.

Финальная структура каждого требования должна содержать только смысловые поля:
- name
- requirements_brief
- check_prompt
- risks
- severity
- check_result_example
- suggested_solutions

Технические поля НЕ заполняй:
- document_category_id
- system_prompt_id
- version
- status
- source

Их добавит Python-код.

Как брать поля из документа:
1. name — короткое название требования или проверки.
   Используй самый короткий и конкретный текст из строки.
   Обычно это колонка "Название риска", но если она заполнена неправильно, определи name по смыслу всей строки.

2. requirements_brief — краткое описание того, что должно быть проверено или соблюдено в договоре.
   Формулируй как проверяемое требование.
   Обычно это берётся из "Описание риска", "Название риска" и "Вопрос к документу".
   Если колонки "Описание риска" и "Название риска" перепутаны, выбери более содержательный текст.

3. check_prompt — вопрос к документу из документа.
   Обычно это колонка "Вопрос к документу".
   Если вопроса нет, сформулируй короткий вопрос по смыслу строки.

4. risks — риск или негативное последствие, которое возникает при отсутствии требования.
   Обычно это колонка "Описание риска" или "Название риска".
   Если отдельного риска нет, используй текст из строки, который описывает негативное последствие.
   Не придумывай новые риски, которых нет в строке.
   

5. severity — степень риска.
   Допустимы только три значения:
   - LOW
   - MEDIUM
   - HIGH

   Правила:
   - "Низкая" → LOW
   - "Средняя" → MEDIUM
   - "Высокая" → HIGH
   - если не знаешь или поле пустое → MEDIUM

6. check_result_example — пример выдачи результата из документа.
   Обычно это колонка "Пример рискованных формулировок".
   Если в документе нет примера, верни пустую строку.

7. suggested_solutions — предложение по исправлению из документа.
   Обычно это колонка "Рекомендация по исправлению".
   Если в документе нет предложения, верни пустую строку.

Правила безопасности JSON:
- Верни только валидный JSON.
- Не используй Markdown.
- Не добавляй поля, которых нет в требуемой структуре.
- Не ставь переносы строк внутри строковых значений.
- Если рекомендация очень длинная, сократи её до главной сути.
- Если пример очень длинный, оставь один короткий пример.
- Внутри строк не используй неэкранированные двойные кавычки.
- Одна строка таблицы обычно соответствует одному требованию.
- Не объединяй разные строки в одно требование.

Формат ответа строго такой:
{{
  "requirements": [
    {{
      "name": "...",
      "requirements_brief": "...",
      "check_prompt": "...",
      "risks": "...",
      "severity": "...",
      "check_result_example": "...",
      "suggested_solutions": "..."
    }}
  ]
}}

Фрагмент документа:
\"\"\"
{chunk.get("text", "")}
\"\"\"
""".strip()


def extract_requirements_from_chunk(chunk: dict) -> list[dict]:
    """Извлекает требования из одного чанка через DeepSeek."""

    prompt = build_prompt_for_chunk(chunk)

    raw_answer = ask_deepseek(
        prompt=prompt,
        model=MODEL_NAME,
        timeout=600,
    )

    data = extract_json_object_from_text(raw_answer)
    return validate_requirements(data, chunk)

## 9. Обработка чанка с fallback

In [9]:
def extract_with_fallback(chunk: dict) -> tuple[list[dict], list[dict]]:
    """
    Сначала пробует обработать лист целиком.
    Если возникает ошибка, делит лист на fallback-подчанки по 20 строк
    и обрабатывает их отдельно.

    Возвращает:
    (requirements, errors)
    """

    local_errors = []

    try:
        requirements = extract_requirements_from_chunk(chunk)
        return requirements, local_errors

    except Exception as primary_exc:
        print("  основная обработка не сработала:", primary_exc)
        print(f"  запускаю fallback по {FALLBACK_ROWS_PER_CHUNK} строк")

        local_errors.append(
            {
                "chunk_id": chunk.get("chunk_id"),
                "source_section": chunk.get("source_section"),
                "stage": "primary_sheet_chunk",
                "error": str(primary_exc),
            }
        )

    fallback_requirements = []
    fallback_chunks = split_markdown_table_chunk_by_rows(
        chunk,
        rows_per_chunk=FALLBACK_ROWS_PER_CHUNK
    )

    for sub_index, subchunk in enumerate(fallback_chunks, start=1):
        print(
            f"  fallback {sub_index}/{len(fallback_chunks)} | "
            f"rows {subchunk.get('row_start')}-{subchunk.get('row_end')}"
        )

        try:
            sub_requirements = extract_requirements_from_chunk(subchunk)
            fallback_requirements.extend(sub_requirements)
            print(f"    найдено требований: {len(sub_requirements)}")

        except Exception as fallback_exc:
            print("    fallback-ошибка:", fallback_exc)
            local_errors.append(
                {
                    "chunk_id": subchunk.get("chunk_id"),
                    "parent_chunk_id": subchunk.get("parent_chunk_id"),
                    "source_section": subchunk.get("source_section"),
                    "row_start": subchunk.get("row_start"),
                    "row_end": subchunk.get("row_end"),
                    "stage": "fallback_rows_chunk",
                    "error": str(fallback_exc),
                }
            )

    return fallback_requirements, local_errors

## 10. Тест на одном чанке

In [10]:
test_chunk = chunks[0]
test_requirements, test_errors = extract_with_fallback(test_chunk)

print(f"Тестовый чанк: {test_chunk.get('source_section')}")
print(f"Строк в листе: {test_chunk.get('rows_count')}")
print(f"Извлечено требований: {len(test_requirements)}")
print(f"Ошибок: {len(test_errors)}")

test_requirements[:3]

Тестовый чанк: поставка, купля-прод.
Строк в листе: 29
Извлечено требований: 29
Ошибок: 0


[{'name': 'Отсутствие условия о том, что неустойка не начисляется в случае несвоевременной уплаты аванса',
  'requirements_brief': 'В договоре должно быть условие, что неустойка за нарушение сроков оплаты не распространяется на авансовые платежи.',
  'check_prompt': 'Распространяется ли условие о неустойки за нарушение сроков оплаты на аванс?',
  'document_category_id': 6,
  'system_prompt_id': 1,
  'version': 1,
  'status': 'published',
  'risks': 'Ответственность за нарушение сроков оплаты Покупателем',
  'severity': 'HIGH',
  'check_result_example': 'Отсутствие условия о том, что неустойка не начисляется в случае несвоевременной уплаты аванса',
  'suggested_solutions': 'Включение условия: "Стороны договорились, что положения пункта 7.5. Договора не распространяет свое действие на авансовые платежи (предоплату)".',
  'source': {'chunk_id': 1,
   'source_section': 'поставка, купля-прод.',
   'sheet_name': 'поставка, купля-прод.',
   'rows_count': 29,
   'chunking_strategy': 'one_sheet

## 11. Обработка всех чанков

In [11]:
all_requirements = []
errors = []

for index, chunk in enumerate(chunks, start=1):
    print(
        f"Чанк {index}/{len(chunks)} | "
        f"Лист: {chunk.get('source_section')} | "
        f"строк: {chunk.get('rows_count')}"
    )

    requirements, local_errors = extract_with_fallback(chunk)

    all_requirements.extend(requirements)
    errors.extend(local_errors)

    print(f"  итог по чанку: требований {len(requirements)}, ошибок {len(local_errors)}")

    checkpoint = {
        "metadata": {
            "model": MODEL_NAME,
            "provider": "deepseek",
            "primary_chunking_strategy": PRIMARY_CHUNKING_STRATEGY,
            "fallback_rows_per_chunk": FALLBACK_ROWS_PER_CHUNK,
            "chunks_total": len(chunks),
            "requirements_total": len(all_requirements),
            "errors_total": len(errors),
            "processed_chunks": index,
        },
        "requirements": all_requirements,
        "errors": errors,
    }

    JSON_CHECKPOINT.write_text(
        json.dumps(checkpoint, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    time.sleep(0.5)

print("Готово.")
print("Всего требований:", len(all_requirements))
print("Ошибок:", len(errors))

Чанк 1/12 | Лист: поставка, купля-прод. | строк: 29
  итог по чанку: требований 29, ошибок 0
Чанк 2/12 | Лист: подряд | строк: 20
  итог по чанку: требований 20, ошибок 0
Чанк 3/12 | Лист: ГПД с ФЛ | строк: 18
  итог по чанку: требований 17, ошибок 0
Чанк 4/12 | Лист: услуги | строк: 18
  итог по чанку: требований 18, ошибок 0
Чанк 5/12 | Лист: аренда | строк: 18
  итог по чанку: требований 18, ошибок 0
Чанк 6/12 | Лист: хранение | строк: 16
  итог по чанку: требований 16, ошибок 0
Чанк 7/12 | Лист: перевозка, экспедиция | строк: 20
  итог по чанку: требований 20, ошибок 0
Чанк 8/12 | Лист: лицензионный | строк: 18
  итог по чанку: требований 1, ошибок 0
Чанк 9/12 | Лист: страхование | строк: 12
  итог по чанку: требований 12, ошибок 0
Чанк 10/12 | Лист: агентский | строк: 16
  итог по чанку: требований 16, ошибок 0
Чанк 11/12 | Лист: благотворит. | строк: 16
  итог по чанку: требований 16, ошибок 0
Чанк 12/12 | Лист: ВЭД особенности | строк: 12
  итог по чанку: требований 12, ошибок 0

## 12. Сохранение финального JSON

In [12]:
result = {
    "metadata": {
        "model": MODEL_NAME,
        "provider": "deepseek",
        "primary_chunking_strategy": PRIMARY_CHUNKING_STRATEGY,
        "fallback_rows_per_chunk": FALLBACK_ROWS_PER_CHUNK,
        "chunks_total": len(chunks),
        "requirements_total": len(all_requirements),
        "errors_total": len(errors),
    },
    "requirements": all_requirements,
    "errors": errors,
}

JSON_OUTPUT.write_text(
    json.dumps(result, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(f"JSON сохранён: {JSON_OUTPUT}")
print("Всего требований:", len(all_requirements))
print("Ошибок:", len(errors))

JSON сохранён: requirements_output_deepseek.json
Всего требований: 195
Ошибок: 0


## 13. Быстрая проверка результата

In [13]:
from collections import Counter

print("Первые 5 требований:")

for req in all_requirements[:5]:
    print("name:", req["name"])
    print("document_category_id:", req["document_category_id"])
    print("system_prompt_id:", req["system_prompt_id"])
    print("version:", req["version"])
    print("status:", req["status"])
    print("severity:", req["severity"])
    print("source:", req["source"].get("chunk_id"), req["source"].get("source_section"), req["source"].get("chunking_strategy"))
    print("brief:", req["requirements_brief"][:300])
    print("-" * 80)

print("Распределение по листам:")
print(Counter(req["source"].get("source_section") for req in all_requirements))

print("Ошибки:")
for error in errors:
    print(error)

Первые 5 требований:
name: Отсутствие условия о том, что неустойка не начисляется в случае несвоевременной уплаты аванса
document_category_id: 6
system_prompt_id: 1
version: 1
status: published
severity: HIGH
source: 1 поставка, купля-прод. one_sheet_one_chunk
brief: Проверить, есть ли в договоре условие о том, что неустойка не начисляется на авансовые платежи.
--------------------------------------------------------------------------------
name: Отсутствие условия о начале течения гарантийного срока
document_category_id: 6
system_prompt_id: 1
version: 1
status: published
severity: HIGH
source: 1 поставка, купля-прод. one_sheet_one_chunk
brief: Проверить, указана ли дата начала исчисления гарантийного срока.
--------------------------------------------------------------------------------
name: Наличие усченного срока возмещения ущерба Поставщику
document_category_id: 6
system_prompt_id: 1
version: 1
status: published
severity: HIGH
source: 1 поставка, купля-прод. one_sheet_one_chunk
br